# Demo 2 · NHL documentation + local Qwen (RAG)

An **LLM** predicts a response from a prompt; it can invent an endpoint or remember an obsolete one. **Retrieval-augmented generation (RAG)** searches documentation first, then supplies relevant passages to the model:

**API documentation → chunks → embeddings → FAISS index → retrieve → prompt Qwen → verify**

An *embedding* represents text as numbers so related passages can be found by similarity. **MiniLM** creates these vectors; **Qwen2.5-1.5B-Instruct** generates the answer. Neither model is trained here.

We will find the NHL play-by-play endpoint and draft a fetch/cache/load function for **MTL–CAR, May 21, 2026 (`2025030311`)**. This one-game example introduces the milestone workflow; the project still requires the specified historical seasons. Continue with [Part 5: PandasAI](05_pandasai.ipynb) for natural-language table queries.

## Setup: choose local or Colab

**Local development (see the [hardware recommendation](../../README_en.md#recommended-hardware-for-notebooks-04-and-05)):** run `uv sync --group llm` from the repository root and select the shared Python 3.11 `.venv` kernel. **Skip the optional installation cell** and continue with the environment check below. You do not need to install packages individually.

**Google Colab | no repository clone or manual uv installation needed:**

1. Open this notebook in Colab. Under **Runtime → Change runtime type**, choose **T4 GPU** and the **2025.07 runtime (Python 3.11)** if available. The default runtime may use a newer Python version; the installation cell cannot change the running interpreter. If no Python 3.11 runtime is available, use the local environment. [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html)
2. In the **optional cell below**, set `INSTALL_COLAB_PACKAGES = True` and run it once. The cell installs `uv`, then uses it to install the libraries into the notebook's current Python. `sys.executable` is that Python's path; `subprocess.check_call` runs a command and stops if it fails. Colab already supplies PyTorch; the cell installs the additional dependencies.
3. Choose **Runtime → Restart session** after installation, even if no restart warning appears. This is needed because NumPy/pandas may already be loaded with different versions.
4. Set the flag back to `False`, then run the **environment check** and continue with **Load Qwen**. Repeat installation when Colab gives you a new runtime; a normal session restart keeps installed packages.

The first model download needs internet; inference then runs in your own runtime, without an API key. CPU mode works but is slower. Use a fresh Colab runtime containing only public NHL files when executing generated code: do not mount Drive or add credentials. For local execution of generated code, use a disposable environment with no access to personal files; a Python virtual environment alone is not a sandbox.

In [ ]:
# OPTIONAL: run once in a fresh Colab runtime; skip during local development.
INSTALL_COLAB_PACKAGES = False  # Set to True to install; reset to False afterward.

import sys
import subprocess

# Detect Colab; otherwise use the local .venv.
try:
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Local environment: skipped. Use uv sync --group llm in your terminal.")
elif not INSTALL_COLAB_PACKAGES:
    print("Installation skipped. Set INSTALL_COLAB_PACKAGES = True if this is a fresh Colab runtime.")
else:
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError("Choose a Python 3.11 Colab runtime before installing these packages.")
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    packages = [
        "ipython>=8,<9",
        "matplotlib-inline<0.2",
        "numpy==1.26.4",
        "pandas==1.5.3",
        "transformers==4.57.6",
        "accelerate>=1,<2",
        "bitsandbytes>=0.45,<1",
        "requests",
        "sentence-transformers==3.4.1",
        "faiss-cpu>=1.8,<2",
        "langchain-text-splitters>=0.3,<0.4",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Choose Runtime → Restart session before continuing.")


### After installation: check the environment

On Colab, **restart the session first** and skip the installation cell on your next run. Locally, run this check after `uv sync --group llm`. Importing NumPy and pandas here also checks that their installed binaries load together.

In [1]:
import sys

if sys.version_info[:2] != (3, 11):
    raise RuntimeError("Use a Python 3.11 kernel: the course .venv locally, or a compatible Colab runtime.")

import numpy as np
import pandas as pd
import torch
from importlib.metadata import version

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "| pandas:", pd.__version__)
print("Transformers:", version("transformers"))
print("Sentence Transformers:", version("sentence-transformers"))
print("CUDA GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Unavailable — CPU/Apple GPU fallback")

Python: 3.11.13
NumPy: 1.26.4 | pandas: 1.5.3
Transformers: 4.57.6
Sentence Transformers: 3.4.1
CUDA GPU: Tesla T4


### Load Qwen

We use Hugging Face pipeline and Qwen chat template. **4-bit** loading stores compressed weights on a CUDA GPU; CPU and Apple GPU runs use ordinary weights. `do_sample=False` uses greedy decoding; outputs can still differ across devices and package versions. The setup code is supplied—focus on the prompts and their results.

**What are we importing?**

- **`torch` (PyTorch)** performs the model's numerical calculations on a CPU or GPU.
- **`transformers`** is Hugging Face's library for loading and using pretrained models.
- **`AutoTokenizer`** loads Qwen's tokenizer: it converts text into numbered *tokens* (words or pieces of words) that the model can process, and converts generated tokens back into text.
- **`AutoModelForCausalLM`** loads the text-generation model. “Causal” means it predicts the next token from the preceding tokens.
- **`BitsAndBytesConfig`** describes how to compress the model's weights into 4-bit values to save GPU memory. It does not train the model.
- **`pipeline`** connects the tokenizer and model into a convenient text-generation tool. This Hugging Face helper is one component of our larger RAG workflow.

**Reading the setup:** `MODEL_ID` selects the model; `from_pretrained(...)` downloads its saved files on the first run and reuses the download cache later. `device` chooses NVIDIA GPU (`cuda`), Apple GPU (`mps`), or CPU. The numeric `dtype` and quantization options control how the model is stored and computed.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
USE_4BIT = True
device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_options = {"dtype": torch.float32 if device == "cpu" else torch.float16}
if device == "cuda":
    model_options["device_map"] = "auto"
    if USE_4BIT:
        model_options["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        )
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_options)
if device != "cuda":
    model = model.to(device)
gen_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

def ask_qwen(prompt, system="You are a helpful assistant.", max_new_tokens=512):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return gen_pipe(text, max_new_tokens=max_new_tokens, do_sample=False,
                    return_full_text=False, pad_token_id=tokenizer.eos_token_id)[0]["generated_text"].strip()

print("Loaded", MODEL_ID, "on", device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cuda:0


Loaded Qwen/Qwen2.5-1.5B-Instruct on cuda


**How `ask_qwen()` works:**

1. Build two messages: `system` gives general instructions; `user` contains our question or task.
2. `apply_chat_template` adds the role markers and formatting Qwen expects. This helps it distinguish instructions from the user's message.
3. `gen_pipe` tokenizes the text, generates tokens, and converts them back to text. `max_new_tokens` limits the answer length; `do_sample=False` picks the most likely next token at each step.
4. `return_full_text=False` returns the new answer rather than repeating the input prompt. The helper extracts `generated_text` from the pipeline's result.

We can reuse this same helper for an explanation or for proposed code—the prompt determines the task.

## 2. Download and index the NHL documentation

We use the [community NHL API reference](https://github.com/Zmalski/NHL-API-Reference) cited by the milestone. It is not an official specification. Reading the raw Markdown avoids scraping navigation menus or a repository landing page.

The cached file makes repeated runs consistent. We keep the **NHL Web API** section, split it into short overlapping chunks, and use **MiniLM + FAISS**, as in the draft. Overlap helps retain context at chunk boundaries. The index stays in memory; keep the cached document and package versions with your experiment records.

**The tools in the next cell:**

- **`Path`** handles local file paths; **`requests`** downloads the documentation, just as it downloaded JSON in Part 1.
- **`RecursiveCharacterTextSplitter`** breaks a long document into smaller passages. It tries paragraph and line boundaries before splitting into smaller pieces.
- **`SentenceTransformer`** loads MiniLM, a model that converts text into an *embedding*: a list of numbers representing its content. Passages with related meanings should have similar vectors.
- **`faiss`** stores and searches those vectors. It finds relevant passages; it does not generate answers.

**Follow the data through the cell:**

1. `web_docs` is the downloaded text, restricted to the NHL Web API section.
2. `chunks` is a list of passages: up to **800 characters** each, with a target overlap of **120 characters** between neighbors. These sizes are characters, not model tokens.
3. `vectors` contains one embedding per chunk. `normalize_embeddings=True` gives each vector unit length; `float32` is the numeric format used here for FAISS.
4. `IndexFlatIP` compares vectors using their dot product. With unit-length vectors, this is **cosine similarity**: a measure of how closely their directions align. `index.add(vectors)` makes the passages searchable.

The index stores vectors; we keep `chunks` alongside it so we can recover the original text after a search.

In [3]:
from pathlib import Path
import requests
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss

DOCS_URL = "https://raw.githubusercontent.com/Zmalski/NHL-API-Reference/main/README.md"
doc_path = Path("data/docs/nhl-api-reference.md")

if not doc_path.exists():
    response = requests.get(DOCS_URL, timeout=30)
    response.raise_for_status()
    doc_path.parent.mkdir(parents=True, exist_ok=True)
    doc_path.write_text(response.text, encoding="utf-8")

web_docs = doc_path.read_text(encoding="utf-8").split("# NHL Web API Documentation", 1)[1]
web_docs = web_docs.split("# NHL Stats API Documentation", 1)[0]

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
chunks = splitter.split_text(web_docs)

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
vectors = embedder.encode(chunks, normalize_embeddings=True).astype("float32")
index = faiss.IndexFlatIP(vectors.shape[1])  # Dot product of unit vectors = cosine similarity.
index.add(vectors)
print(len(chunks), "chunks indexed from", DOCS_URL)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

84 chunks indexed from https://raw.githubusercontent.com/Zmalski/NHL-API-Reference/main/README.md


## 3. Retrieve first, then generate

Read the passages **before** reading Qwen's answer. Does retrieval find the endpoint? A wrong answer can come from poor retrieval, poor generation, or both.

**What the search does:**

- Encode the question with the **same MiniLM model** used for the documentation, so their vectors can be compared.
- `index.search(..., k=3)` returns the three closest vectors. `scores` contains their similarities; `positions` contains their locations in our `chunks` list. A similarity score is not a probability that a passage answers the question correctly.
- Use those positions to recover the passages and join them into `context`, with labels `[1]`, `[2]`, and `[3]` for citations.
- The `assert` checks that the expected play-by-play endpoint appears somewhere in the retrieved text. At this stage, **Qwen has not answered the question yet**.

In [4]:
question = "Which endpoint retrieves play-by-play information for a specific game ID?"
query_vector = embedder.encode([question], normalize_embeddings=True).astype("float32")
scores, positions = index.search(query_vector, k=3)
context = "\n\n".join(f"[{i}] {chunks[position]}" for i, position in enumerate(positions[0], 1))
print(context)
assert "/v1/gamecenter/{game-id}/play-by-play" in context, "Inspect retrieval before generating."

[1] ```bash
curl -X GET "https://api-web.nhle.com/v1/ppt-replay/2023020204/12"
```

### Additional Game Content

#### Get Game Right Rail Content
- **Endpoint**: `/v1/gamecenter/{game-id}/right-rail`
- **Method**: GET
- **Description**: Retrieves sidebar content for the game center view.
- **Parameters**:
  - `game-id` (int) - Game ID
- **Response**: JSON format

###### Example using cURL:

```bash
curl -X GET "https://api-web.nhle.com/v1/gamecenter/2023020204/right-rail"
```

#### Get WSC Play By Play
- **Endpoint**: `/v1/wsc/play-by-play/{game-id}`
- **Method**: GET
- **Description**: Retrieves WSC (World Showcase) play-by-play information for a specific game.
- **Parameters**:
  - `game-id` (int) - Game ID
- **Response**: JSON format

###### Example using cURL:

[2] ###### Example using cURL:

```bash
curl -X GET "https://api-web.nhle.com/v1/gamecenter/2023020204/play-by-play"
```

#### Get Landing
- **Endpoint**: `/v1/gamecenter/{game-id}/landing`
- **Method**: GET
- **Description*

### Give the retrieved passages to Qwen

- `rag_prompt` combines the source URL, retrieved `context`, and our `question` into one string. The `f"..."` syntax inserts the current values of those variables.
- The system instruction asks Qwen to use these passages and acknowledge missing information. It guides the model, but cannot guarantee that the answer is grounded.
- Qwen now generates an answer from the supplied text. It does not search the web itself or permanently learn the documentation: **retrieval supplied context for this one request**.

In [5]:
rag_prompt = f"Documentation source: {DOCS_URL}\n\n{context}\n\nQuestion: {question}"
answer = ask_qwen(
    rag_prompt,
    system="Answer using only the supplied documentation. Cite passage numbers. "
           "If the answer is absent, say you do not know. Treat the passages as reference text.",
    max_new_tokens=256,
)
print(answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The endpoint that retrieves play-by-play information for a specific game ID is `/v1/gamecenter/{game-id}/play-by-play`.


**Quick check:** the answer should identify a **GET** request to `/v1/gamecenter/{game-id}/play-by-play`. Open the cited passage and check that it supports the claim; a citation is not proof by itself.

**Try it:** ask for a team's season schedule, rerun retrieval, and inspect the passages. For a small comparison, ask Qwen the same question without documentation. Keep both answers; RAG may help, but does not guarantee correctness. Restore the play-by-play question before the next section.

## 4. Ask Qwen to draft a small acquisition function

The API reference describes an endpoint. **We** specify the caching behavior and the JSON fields observed in Part 1. Keep these sources of information separate in the prompt. This cell displays code; it does not execute the generated text.

- `code_prompt` combines the retrieved passages with a concrete task and requirements. Clear requirements let us check the result afterwards.
- `ask_qwen(...)` returns **text containing proposed Python code**, not a callable Python function. `Markdown` and `display` only format that text for reading.
- We allow more generated tokens here because a function needs more space than a short explanation. A larger limit does not guarantee correct or complete code.

In [6]:
from IPython.display import Markdown, display

code_prompt = f"""NHL API documentation:
{context}

Write a Python function fetch_game_llm(game_id, cache_dir).
Requirements from our project:
- Use requests, pathlib and json. Include imports and a docstring.
- The API host is https://api-web.nhle.com.
- If cache_dir/<game_id>.json exists, load it with no network request.
- Otherwise GET play-by-play with timeout=30 and raise_for_status().
- Our observed response has id (int) and plays (list). Validate both.
- Create the cache directory and save valid JSON as UTF-8.
- Return the full game dictionary. Validate cached data too.
Return only the function and imports in a Python code block.
"""
generated_code = ask_qwen(code_prompt, system="You write concise, correct Python code.", max_new_tokens=768)
display(Markdown(generated_code))

```python
import requests
from pathlib import Path
import json

def fetch_game_llm(game_id: int, cache_dir: str):
    """
    Fetches play-by-play information for a given game ID.

    Parameters:
    game_id (int): Unique identifier of the game.
    cache_dir (str): Directory where the cache file will be saved.

    Returns:
    dict: A dictionary containing the full game details including play-by-play.
    """

    # Construct the URL based on the endpoint
    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"

    # Check if the cache file already exists
    cache_path = Path(cache_dir).joinpath(f"{game_id}.json")
    
    try:
        # Load from cache if available
        with open(cache_path, 'r', encoding='utf-8') as cache_file:
            return json.load(cache_file)
        
    except FileNotFoundError:
        pass
    
    # Fetch data from the server
    response = requests.get(url, timeout=30)

    # Raise an error if the status code indicates failure
    response.raise_for_status()

    # Parse the response into a dictionary
    data = response.json()
    
    # Save the data to cache
    with open(cache_path, 'w', encoding='utf-8') as cache_file:
        json.dump(data, cache_file, indent=4)
    
    # Return the full game dictionary
    return data

# Example usage
if __name__ == "__main__":
    game_id = 2023020204
    cache_dir = "/path/to/cache"
    fetched_data = fetch_game_llm(game_id, cache_dir)
    print(fetched_data)
```

### Your turn: review and compare

Check the proposed URL, imports, HTTP errors, validation, and cache behavior. A small model may produce incomplete or incorrect code: record what failed and revise the prompt.

The next cell reuses **Part 1's reference workflow** to obtain the raw JSON. Copy its cached file to the displayed path if you already have it. For the milestone's *manual implementation*, write your own code before using an LLM and disclose AI assistance; the supplied teaching example is not your team's manual solution.

**Reminder of the reference workflow below:**

- If `raw_path` already exists, read the saved JSON. Otherwise, request the game, check the HTTP response and key fields, then save it.
- `json.dumps` converts a Python object into JSON text; `json.loads` converts JSON text back into a Python object.
- `game` is the reference dictionary we will compare against the generated function's result. This separates checking the LLM's code from trusting its explanation.

In [7]:
import json
from pathlib import Path
import requests

GAME_ID = 2025030311
raw_path = Path("data/raw") / f"{GAME_ID}.json"
print("Cached game:", raw_path.resolve())
if not raw_path.exists():
    response = requests.get(
        f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play", timeout=30,
    )
    response.raise_for_status()
    downloaded = response.json()
    assert downloaded["id"] == GAME_ID and isinstance(downloaded["plays"], list)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(json.dumps(downloaded), encoding="utf-8")
game = json.loads(raw_path.read_text(encoding="utf-8"))
assert game["id"] == GAME_ID and isinstance(game["plays"], list)
print(game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"], game["gameDate"])

Cached game: /content/data/raw/2025030311.json
MTL at CAR 2026-05-21


### Turn the proposed text into a function

After reviewing the generated code, paste its imports and `def fetch_game_llm(...)` into the next cell, replacing the placeholder. Running that cell defines the function; the following cell calls it. Leaving `None` in place skips the comparison.

In [ ]:
# In a fresh runtime containing only public data, paste the REVIEWED function below.
# Remove this placeholder when replacing it with def fetch_game_llm(...).
fetch_game_llm = None

### Check fetching and caching without downloading again

We control the network response so both implementations can be compared using the same game data:

- **`TemporaryDirectory`** creates a scratch folder and removes it afterwards. The generated function must create the `raw` subfolder itself.
- **`Mock`** acts as a fake HTTP response: calling its `.json()` returns our reference `game`.
- **`patch`** temporarily replaces `requests.get`. On the first call it returns the fake response; on the second it raises an error if the function tries to use the network instead of its cache.
- The **assertions** check the requested URL and timeout, the HTTP-status check, the saved JSON, and both returned dictionaries. A failure tells us which behavior to investigate; fluent-looking code alone is not evidence that the pipeline works.

In [ ]:
from tempfile import TemporaryDirectory
from unittest.mock import Mock, patch

if fetch_game_llm is None:
    print("Review and paste the generated function above to run the comparison.")
else:
    with TemporaryDirectory() as temp_dir:
        folder = Path(temp_dir) / "raw"  # The function must create this directory.
        response = Mock()
        response.json.return_value = game
        with patch("requests.get", return_value=response) as get:
            first = fetch_game_llm(GAME_ID, folder)
        get.assert_called_once_with(
            f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play", timeout=30,
        )
        response.raise_for_status.assert_called_once()
        saved = json.loads((Path(folder) / f"{GAME_ID}.json").read_text(encoding="utf-8"))
        with patch("requests.get", side_effect=AssertionError("Cache was ignored")):
            second = fetch_game_llm(GAME_ID, folder)
        assert first == second == saved == game
    print("Returned data and cache behavior match the reference JSON.")

This check simulates the initial HTTP response and forbids networking on the second call. It checks returned data **and** caching, but does not prove that every failure case works. Add a check for an HTTP error or a cached file containing the wrong game.

**Sources:** [Qwen model card](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) · [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) · [NHL reference](https://github.com/Zmalski/NHL-API-Reference). Model outputs are deliberately left empty for students to generate and evaluate.